# FUNSD Handwriting Augmentation Pipeline (v2 — High Quality)

Replaces printed/typed text in FUNSD documents with **synthetically rendered handwriting**,
preserving all bounding-box annotations exactly.

### Setup before running
1. **Add the FUNSD dataset** → *Add Data → Search Datasets → "FUNSD"*
2. Make sure **Internet is ON**
3. Run all cells top-to-bottom
4. Final cell zips the output for download

### What's improved in v2
- Fixes `trdg` crash (`arabic_reshaper` installed correctly)
- Downloads a real **Google handwriting font** (Caveat) as automatic fallback
- **Smart background sampling** — matches actual paper colour per-region (no more ugly gray boxes)
- **Per-character jitter** for natural baseline variation
- Ink colour sampled from real pen-ink spectrum (dark blue / graphite)
- Hard alpha threshold so ink is crisp and legible

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — Install all dependencies
# ═══════════════════════════════════════════════════════════════════
import subprocess, sys

def run(cmd, label=""):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tag = f"[{label}] " if label else ""
    if r.returncode == 0:
        print(f"{tag}OK")
    else:
        print(f"{tag}FAILED:\n{r.stderr[-1500:]}")
    return r.returncode

# arabic_reshaper is a direct trdg dependency that was missing → caused the crash
run("pip install arabic_reshaper python-bidi -q",  "arabic_reshaper")
run("pip install trdg --no-deps -q",               "trdg")
run("pip install tqdm -q",                         "tqdm")

# Verify
for pkg in ["trdg", "arabic_reshaper", "cv2", "numpy", "PIL", "tqdm"]:
    try:
        __import__(pkg if pkg != "PIL" else "PIL.Image")
        print(f"  {pkg}: ✓")
    except ImportError as e:
        print(f"  {pkg}: ✗ ({e})")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — Download handwriting fonts (used as high-quality fallback)
# ═══════════════════════════════════════════════════════════════════
import requests
from pathlib import Path

FONT_DIR = Path("/kaggle/working/hw_fonts")
FONT_DIR.mkdir(exist_ok=True)

# Google Fonts — open-source handwriting fonts
FONTS_TO_FETCH = {
    "Caveat-Regular.ttf":       "https://github.com/google/fonts/raw/main/ofl/caveat/Caveat%5Bwght%5D.ttf",
    "Caveat-Bold.ttf":          "https://github.com/google/fonts/raw/main/ofl/caveat/Caveat%5Bwght%5D.ttf",
    "IndieFlower.ttf":          "https://github.com/google/fonts/raw/main/ofl/indieflower/IndieFlower-Regular.ttf",
    "DancingScript.ttf":        "https://github.com/google/fonts/raw/main/ofl/dancingscript/DancingScript%5Bwght%5D.ttf",
    "PatrickHand.ttf":          "https://github.com/google/fonts/raw/main/ofl/patrickhand/PatrickHand-Regular.ttf",
    "Kalam-Regular.ttf":        "https://github.com/google/fonts/raw/main/ofl/kalam/Kalam-Regular.ttf",
    "Kalam-Bold.ttf":           "https://github.com/google/fonts/raw/main/ofl/kalam/Kalam-Bold.ttf",
    "CoveredByYourGrace.ttf":   "https://github.com/google/fonts/raw/main/ofl/coveredbyyourgrace/CoveredByYourGrace-Regular.ttf",
}

downloaded = []
for fname, url in FONTS_TO_FETCH.items():
    dest = FONT_DIR / fname
    if dest.exists():
        print(f"  {fname}: already cached")
        downloaded.append(str(dest))
        continue
    try:
        resp = requests.get(url, timeout=20)
        if resp.status_code == 200 and len(resp.content) > 1000:
            dest.write_bytes(resp.content)
            print(f"  {fname}: downloaded ({len(resp.content)//1024} KB)")
            downloaded.append(str(dest))
        else:
            print(f"  {fname}: HTTP {resp.status_code} (skip)")
    except Exception as exc:
        print(f"  {fname}: failed ({exc})")

print(f"\n{len(downloaded)} handwriting fonts ready in {FONT_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — Configuration  ← edit this
# ═══════════════════════════════════════════════════════════════════
import os
from pathlib import Path

# Auto-detect FUNSD root
def find_funsd_root():
    for p in sorted(Path("/kaggle/input").rglob("training_data")):
        if (p / "annotations").exists() and (p / "images").exists():
            return p.parent
    return None

FUNSD_ROOT = find_funsd_root() or Path("/kaggle/input/funsd-dataset")
print(f"FUNSD root : {FUNSD_ROOT}")

OUTPUT_ROOT  = Path("/kaggle/working/funsd_handwritten")
SPLITS       = ["training_data", "testing_data"]

# ── Rendering backend ──────────────────────────────────────────────
# 'trdg'        → TextRecognitionDataGenerator (best, uses handwriting fonts)
# 'pillow_hw'   → Pillow + downloaded Google fonts (good quality, always works)
# 'scrabblegan' → GAN-based (needs separate weights dataset)
STYLE        = "trdg"

# Number of augmented variants per document
N_VARIANTS   = 1

# How to erase original text before pasting:
#   'smart'   → sample real paper colour from margins (RECOMMENDED)
#   'fill'    → solid background colour
#   'inpaint' → OpenCV inpainting (slowest, most realistic)
ERASE_METHOD = "smart"

# Fallback solid colour (only used if ERASE_METHOD='fill')
FILL_COLOR   = (248, 244, 236)

# trdg: max word skew angle in degrees
SKEW_ANGLE   = 2

# ScrabbleGAN paths (only when STYLE='scrabblegan')
SCRABBLEGAN_REPO    = "/kaggle/input/scrabblegan/convolutional-handwriting-gan"
SCRABBLEGAN_WEIGHTS = "/kaggle/input/scrabblegan/scrabblegan_weights.pth"

RANDOM_SEED  = 42

print(f"Style      : {STYLE}")
print(f"Variants   : {N_VARIANTS}")
print(f"Erase      : {ERASE_METHOD}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Renderer classes (v2 — high quality)
# ═══════════════════════════════════════════════════════════════════
import random, sys, warnings
from pathlib import Path
from typing import List, Optional, Tuple

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

warnings.filterwarnings("ignore")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# ─────────────────────────────────────────────────────────────────
#  Renderer 1 : TRDG  (fixed — installs arabic_reshaper first)
# ─────────────────────────────────────────────────────────────────
class TRDGRenderer:
    """Renders words using TextRecognitionDataGenerator handwriting fonts."""

    def __init__(self, skewing_angle: int = 2, blur: int = 0,
                 font_dir: Optional[str] = None):
        from trdg.generators import GeneratorFromStrings
        self._Gen = GeneratorFromStrings
        self.skewing_angle = skewing_angle
        self.blur = blur
        self.font_dir = font_dir
        print("TRDGRenderer ready")

    def render_word(self, text: str, target_h: int, target_w: int) -> Image.Image:
        blank = Image.new("RGB", (max(target_w, 4), max(target_h, 4)), (255, 255, 255))
        if not text.strip():
            return blank
        try:
            kwargs = dict(
                strings=[text], count=1,
                random_blur=self.blur > 0, blur=self.blur,
                random_skew=True, skewing_angle=self.skewing_angle,
                background_type=1,   # white background
                language="en", handwritten=True,
                size=max(target_h, 20), image_mode="RGB",
            )
            if self.font_dir:
                kwargs["fonts"] = [self.font_dir]
            gen = self._Gen(**kwargs)
            img, _ = next(iter(gen))
            return img.convert("RGB").resize(
                (max(target_w, 2), max(target_h, 2)), Image.LANCZOS
            )
        except Exception as exc:
            print(f"  [trdg] '{text}': {exc}", file=sys.stderr)
            return blank


# ─────────────────────────────────────────────────────────────────
#  Renderer 2 : High-Quality Pillow + Google Handwriting Fonts
# ─────────────────────────────────────────────────────────────────
class PillowHWRenderer:
    """
    Much-improved Pillow renderer:
    • Uses real Google handwriting fonts (Kalam, Caveat, IndieFlower …)
    • Per-character vertical jitter for natural baseline variation
    • Ink colour sampled from realistic pen/graphite spectrum
    • Renders at 4× scale, downsampled for crisp antialiasing
    • Hard alpha threshold so ink looks clean, not smeared
    """

    # Fallback system fonts if downloads failed
    _SYSTEM_FONTS = [
        "/usr/share/fonts/truetype/liberation/LiberationMono-Regular.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/freefont/FreeMono.ttf",
    ]

    # Ink colours: (R, G, B) tuples that look like real pen ink
    INK_PALETTES = [
        # dark blue-black ballpoint
        [(15, 18, 55), (20, 25, 70), (10, 12, 45)],
        # graphite pencil
        [(55, 55, 58), (45, 45, 48), (65, 65, 68)],
        # dark navy
        [(10, 20, 80), (8,  15, 65), (15, 28, 90)],
    ]

    def __init__(self, font_paths: Optional[List[str]] = None):
        # Collect available fonts, prefer downloaded Google fonts
        self._fonts_paths: List[str] = []
        for p in (font_paths or []):
            if Path(p).exists():
                self._fonts_paths.append(p)
        if not self._fonts_paths:
            for p in self._SYSTEM_FONTS:
                if Path(p).exists():
                    self._fonts_paths.append(p)
        if not self._fonts_paths:
            self._fonts_paths = [None]   # ImageFont.load_default()

        print(f"PillowHWRenderer ready — {len([p for p in self._fonts_paths if p])} fonts")
        self._cache: dict = {}

    # ── internal helpers ──────────────────────────────────────────
    def _get_font(self, path: Optional[str], size: int) -> ImageFont.FreeTypeFont:
        key = (path, size)
        if key not in self._cache:
            try:
                self._cache[key] = ImageFont.truetype(path, size) if path else ImageFont.load_default()
            except Exception:
                self._cache[key] = ImageFont.load_default()
        return self._cache[key]

    def _pick_ink(self) -> Tuple[int, int, int]:
        palette = random.choice(self.INK_PALETTES)
        base = random.choice(palette)
        # tiny random variation per word
        return tuple(max(0, min(255, c + random.randint(-8, 8))) for c in base)

    # ── public API ────────────────────────────────────────────────
    def render_word(self, text: str, target_h: int, target_w: int) -> Image.Image:
        blank = Image.new("RGB", (max(target_w, 4), max(target_h, 4)), (255, 255, 255))
        if not text.strip():
            return blank

        SCALE = 4  # render at 4× for quality
        H = max(target_h, 10) * SCALE
        W = max(target_w, 10) * SCALE

        font_path = random.choice(self._fonts_paths)
        # Font size: ~80% of height gives good cap-height
        font_size = max(int(target_h * 0.80 * SCALE), 10)
        font = self._get_font(font_path, font_size)
        ink  = self._pick_ink()

        canvas = Image.new("RGB", (W * 2, H), (255, 255, 255))   # wider canvas, crop later
        draw   = ImageDraw.Draw(canvas)

        # ── per-character rendering with vertical jitter ──────────
        x_cursor = int(H * 0.05)   # small left margin
        max_jitter = max(int(H * 0.08), 1)

        for ch in text:
            jitter_y = random.randint(-max_jitter, max_jitter)
            draw.text(
                (x_cursor, int(H * 0.05) + jitter_y),
                ch, font=font, fill=ink
            )
            # Advance x by character width
            try:
                bb = font.getbbox(ch)
                ch_w = (bb[2] - bb[0]) + random.randint(-1, 2)
            except Exception:
                ch_w = font_size // 2
            x_cursor += max(ch_w, 1)

        # Crop to actual text extent
        canvas_np = np.array(canvas)
        mask = canvas_np.min(axis=2) < 240   # non-white pixels
        rows = np.any(mask, axis=1)
        cols = np.any(mask, axis=0)
        if rows.any() and cols.any():
            r0, r1 = np.where(rows)[0][[0, -1]]
            c0, c1 = np.where(cols)[0][[0, -1]]
            pad = max(int(H * 0.05), 2)
            r0 = max(0, r0 - pad);  r1 = min(canvas_np.shape[0] - 1, r1 + pad)
            c0 = max(0, c0 - pad);  c1 = min(canvas_np.shape[1] - 1, c1 + pad)
            canvas = Image.fromarray(canvas_np[r0:r1+1, c0:c1+1])

        # Slight rotation for realism
        angle = random.uniform(-1.5, 1.5)
        canvas = canvas.rotate(angle, fillcolor=(255, 255, 255), expand=False)

        # Subtle Gaussian blur to soften font edges
        canvas = canvas.filter(ImageFilter.GaussianBlur(radius=0.4 * SCALE))

        # Downsample to target size
        return canvas.resize((max(target_w, 2), max(target_h, 2)), Image.LANCZOS)


# ─────────────────────────────────────────────────────────────────
#  Renderer 3 : ScrabbleGAN (GAN-based, best quality)
# ─────────────────────────────────────────────────────────────────
class ScrabbleGANRenderer:
    """Renders words using the ScrabbleGAN GAN model."""

    def __init__(self, model_root: str, weights_path: str, device: str = "cpu"):
        import torch
        sys.path.insert(0, model_root)
        from data import alphabets
        from models import ScrabbleGAN as SGModel  # type: ignore
        self.torch = torch
        self.device = torch.device(device)
        self.alphabet = alphabets.alphabet
        state = torch.load(weights_path, map_location=self.device)
        self.model = SGModel(state["opt"]).to(self.device)
        self.model.load_state_dict(state["state_dict"])
        self.model.eval()
        print(f"ScrabbleGANRenderer ready — device: {device}")

    def render_word(self, text: str, target_h: int, target_w: int) -> Image.Image:
        import torch
        blank = Image.new("RGB", (max(target_w, 4), max(target_h, 4)), (255, 255, 255))
        encoded = [self.alphabet.index(c) for c in text if c in self.alphabet]
        if not encoded:
            return blank
        label = torch.LongTensor(encoded).unsqueeze(0).to(self.device)
        noise = torch.randn(1, 128).to(self.device)
        with torch.no_grad():
            fake = self.model.G(noise, label).squeeze().cpu().numpy()
        mn, mx = fake.min(), fake.max()
        fake = ((fake - mn) / (mx - mn + 1e-8) * 255).astype(np.uint8)
        return Image.fromarray(fake).convert("RGB").resize(
            (max(target_w, 2), max(target_h, 2)), Image.LANCZOS
        )


# ─────────────────────────────────────────────────────────────────
#  Factory
# ─────────────────────────────────────────────────────────────────
def build_renderer(style: str):
    if style == "trdg":
        try:
            return TRDGRenderer(skewing_angle=SKEW_ANGLE)
        except Exception as e:
            print(f"[warn] trdg failed: {e} → falling back to PillowHW")
            return PillowHWRenderer(font_paths=downloaded)

    elif style == "scrabblegan":
        try:
            import torch
            dev = "cuda" if torch.cuda.is_available() else "cpu"
            return ScrabbleGANRenderer(SCRABBLEGAN_REPO, SCRABBLEGAN_WEIGHTS, dev)
        except Exception as e:
            print(f"[warn] ScrabbleGAN failed: {e} → falling back to trdg")
            return build_renderer("trdg")

    else:  # 'pillow_hw'
        return PillowHWRenderer(font_paths=downloaded)


RENDERER = build_renderer(STYLE)
print(f"\nActive renderer: {RENDERER.__class__.__name__}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Augmentation engine (v2)
# ═══════════════════════════════════════════════════════════════════
import json


def sample_paper_color(img_np: np.ndarray, x0: int, y0: int, x1: int, y1: int,
                        margin: int = 8) -> Tuple[int, int, int]:
    """
    Sample the median colour from a margin AROUND the bounding box.
    This gives us the real paper/background colour at that location
    so the erase step is invisible.
    """
    ih, iw = img_np.shape[:2]
    # Build a ring of pixels around the box
    samples = []
    for dy in range(-margin, 0):
        row = y0 + dy
        if 0 <= row < ih:
            samples.extend(img_np[row, max(0,x0):min(iw,x1)].tolist())
    for dy in range(1, margin + 1):
        row = y1 + dy
        if 0 <= row < ih:
            samples.extend(img_np[row, max(0,x0):min(iw,x1)].tolist())

    if not samples:
        # Fallback: sample from top-left corner of image (usually blank paper)
        corner = img_np[:min(20, ih), :min(20, iw)]
        return tuple(int(v) for v in np.median(corner.reshape(-1, 3), axis=0))

    arr = np.array(samples, dtype=np.float32)
    # Use the 80th-percentile brightness so we pick paper, not ink
    bright = arr.mean(axis=1)
    bright_idx = bright > np.percentile(bright, 40)
    bright_pixels = arr[bright_idx] if bright_idx.any() else arr
    median_color = np.median(bright_pixels, axis=0)
    return tuple(int(v) for v in median_color)


def paste_handwritten_words(
    base_image: Image.Image,
    annotation: dict,
    renderer,
    erase_method: str = "smart",
    fill_color: Tuple[int, int, int] = (248, 244, 236),
    alpha_threshold: float = 0.15,
) -> Image.Image:
    """
    For every word in the FUNSD annotation:
      1. Erase the original text in the bounding box.
      2. Render the word as handwriting.
      3. Composite ink onto the (now-erased) background.

    Parameters
    ----------
    erase_method    : 'smart' | 'fill' | 'inpaint'
    alpha_threshold : ink pixels below this alpha are treated as paper (keeps edges clean)
    """
    img_np = np.array(base_image.convert("RGB"))
    ih, iw = img_np.shape[:2]

    for field in annotation.get("form", []):
        for word_obj in field.get("words", []):
            text = word_obj.get("text", "").strip()
            box  = word_obj.get("box", [])
            if not text or len(box) != 4:
                continue

            # Normalise + clamp bounding box
            x0, y0, x1, y1 = sorted([box[0], box[2]])[0], sorted([box[1], box[3]])[0], \
                              sorted([box[0], box[2]])[1], sorted([box[1], box[3]])[1]
            x0 = max(0, min(x0, iw - 2)); x1 = max(x0 + 2, min(x1, iw))
            y0 = max(0, min(y0, ih - 2)); y1 = max(y0 + 2, min(y1, ih))
            w, h = x1 - x0, y1 - y0

            # ── Step 1: erase original text ────────────────────────
            if erase_method == "smart":
                bg = sample_paper_color(img_np, x0, y0, x1, y1)
                img_np[y0:y1, x0:x1] = bg
            elif erase_method == "inpaint":
                mask = np.zeros((ih, iw), dtype=np.uint8)
                mask[y0:y1, x0:x1] = 255
                img_np = cv2.inpaint(img_np, mask, 3, cv2.INPAINT_TELEA)
            else:  # 'fill'
                img_np[y0:y1, x0:x1] = fill_color

            # ── Step 2: render handwritten word ────────────────────
            hw_img = renderer.render_word(text, h, w)
            hw_np  = np.array(hw_img.convert("RGB"))
            if hw_np.shape[:2] != (h, w):
                hw_np = cv2.resize(hw_np, (w, h), interpolation=cv2.INTER_LANCZOS4)

            # ── Step 3: composite with hard alpha threshold ─────────
            # Convert white (paper) → transparent, ink → opaque
            gray  = cv2.cvtColor(hw_np, cv2.COLOR_RGB2GRAY).astype(np.float32)
            alpha = ((255.0 - gray) / 255.0)  # 0=white paper, 1=black ink
            # Hard threshold: pixels below threshold become fully transparent
            alpha = np.where(alpha < alpha_threshold, 0.0, alpha)
            # Optional: sharpen the alpha slightly
            alpha = np.clip(alpha * 1.3, 0.0, 1.0)
            alpha3 = np.stack([alpha] * 3, axis=-1)

            bg_patch = img_np[y0:y1, x0:x1].astype(np.float32)
            fg_patch = hw_np.astype(np.float32)
            blended  = (alpha3 * fg_patch + (1.0 - alpha3) * bg_patch).clip(0, 255).astype(np.uint8)
            img_np[y0:y1, x0:x1] = blended

    return Image.fromarray(img_np)


print("paste_handwritten_words() v2 defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — Preview on one document (side-by-side)
# ═══════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

def preview_one(split="training_data", doc_index=0):
    ann_dir   = FUNSD_ROOT / split / "annotations"
    img_dir   = FUNSD_ROOT / split / "images"
    json_files = sorted(ann_dir.glob("*.json"))
    if not json_files:
        print(f"No annotation files in {ann_dir}"); return

    doc_index = doc_index % len(json_files)
    json_path = json_files[doc_index]
    stem = json_path.stem

    img_path = next(
        (img_dir / (stem + ext) for ext in (".png",".jpg",".jpeg")
         if (img_dir / (stem + ext)).exists()), None
    )
    if img_path is None:
        print(f"No image for {stem}"); return

    with open(json_path) as f:
        ann = json.load(f)

    original  = Image.open(img_path).convert("RGB")
    augmented = paste_handwritten_words(
        original, ann, RENDERER,
        erase_method=ERASE_METHOD,
        fill_color=FILL_COLOR,
    )

    fig, axes = plt.subplots(1, 2, figsize=(18, 11))
    axes[0].imshow(original);  axes[0].set_title("Original",  fontsize=15, fontweight="bold")
    axes[0].axis("off")
    axes[1].imshow(augmented); axes[1].set_title("Handwritten Augmentation", fontsize=15, fontweight="bold")
    axes[1].axis("off")
    plt.tight_layout()
    plt.savefig("/kaggle/working/preview.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved: /kaggle/working/preview.png  (doc: {stem})")


preview_one("training_data", doc_index=0)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — Run full pipeline
# ═══════════════════════════════════════════════════════════════════
from tqdm.notebook import tqdm

def process_split(split_name: str):
    split_dir     = FUNSD_ROOT / split_name
    out_split_dir = OUTPUT_ROOT / split_name
    if not split_dir.exists():
        print(f"[skip] {split_dir} not found"); return

    ann_dir     = split_dir     / "annotations"
    img_dir     = split_dir     / "images"
    out_ann_dir = out_split_dir / "annotations"
    out_img_dir = out_split_dir / "images"
    out_ann_dir.mkdir(parents=True, exist_ok=True)
    out_img_dir.mkdir(parents=True, exist_ok=True)

    json_files = sorted(ann_dir.glob("*.json"))
    print(f"\n[{split_name}] {len(json_files)} documents")

    for json_path in tqdm(json_files, desc=split_name):
        stem = json_path.stem
        img_path = next(
            (img_dir / (stem + ext) for ext in (".png",".jpg",".jpeg",".tif")
             if (img_dir / (stem + ext)).exists()), None
        )
        if img_path is None:
            tqdm.write(f"  [warn] no image for {stem}"); continue

        with open(json_path) as f:
            annotation = json.load(f)

        base_img = Image.open(img_path).convert("RGB")

        for v in range(N_VARIANTS):
            suffix = f"_v{v}" if N_VARIANTS > 1 else ""
            aug_img = paste_handwritten_words(
                base_img, annotation, RENDERER,
                erase_method=ERASE_METHOD,
                fill_color=FILL_COLOR,
            )
            out_img_path  = out_img_dir / (stem + suffix + img_path.suffix)
            out_json_path = out_ann_dir / (stem + suffix + ".json")
            aug_img.save(out_img_path)
            with open(out_json_path, "w") as f:
                json.dump(annotation, f, indent=2)


OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for split in SPLITS:
    process_split(split)

img_count = sum(1 for _ in OUTPUT_ROOT.rglob("*.png")) + sum(1 for _ in OUTPUT_ROOT.rglob("*.jpg"))
ann_count = sum(1 for _ in OUTPUT_ROOT.rglob("*.json"))
print(f"\nDone!  Images: {img_count}  |  Annotations: {ann_count}")
print(f"Output: {OUTPUT_ROOT}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — Zip output for download
# ═══════════════════════════════════════════════════════════════════
import shutil
from pathlib import Path

zip_base = "/kaggle/working/funsd_handwritten"
print("Zipping ...")
shutil.make_archive(zip_base, "zip",
                    root_dir=str(OUTPUT_ROOT.parent),
                    base_dir=OUTPUT_ROOT.name)
size_mb = Path(zip_base + ".zip").stat().st_size / 1e6
print(f"Created: {zip_base}.zip  ({size_mb:.1f} MB)")
print("Download via the Kaggle → Output panel (right sidebar).")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9 — Sample grid of results (4 random docs)
# ═══════════════════════════════════════════════════════════════════
import random as _rnd
import matplotlib.pyplot as plt

aug_imgs = sorted((OUTPUT_ROOT / "training_data" / "images").glob("*.*"))
ori_imgs = {p.stem: p for p in (FUNSD_ROOT / "training_data" / "images").glob("*.*")}

n_show = min(4, len(aug_imgs))
picks  = _rnd.sample(aug_imgs, n_show)

fig, axes = plt.subplots(n_show, 2, figsize=(16, 5 * n_show))
if n_show == 1: axes = [axes]

for row, aug_p in enumerate(picks):
    stem  = aug_p.stem.split("_v")[0]
    ori_p = ori_imgs.get(stem)
    if ori_p:
        axes[row][0].imshow(Image.open(ori_p));  axes[row][0].set_title(f"Original: {stem}",    fontsize=10); axes[row][0].axis("off")
    axes[row][1].imshow(Image.open(aug_p));  axes[row][1].set_title(f"Handwritten: {aug_p.name}", fontsize=10); axes[row][1].axis("off")

plt.suptitle("FUNSD Handwriting Augmentation — Sample Results", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("/kaggle/working/samples_grid.png", dpi=90, bbox_inches="tight")
plt.show()
print("Grid saved: /kaggle/working/samples_grid.png")